In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
# Migrates several single Weaviate collections (identical schema) into one global
# multi-tenant collection: each source collection becomes a tenant.
#
# Steps:
#   1. Connect to the source and target instances.
#   2. Use the first source collection as a schema template and create a global
#      multi-tenant collection.
#   3. Map each source collection to a tenant and copy its objects across.

import logging
import sys

import weaviate
import weaviate.classes as wvc
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Vectorizers
from tqdm import tqdm

# --- CONFIGURATION ---
SRC_URL = "<SOURCE-WEAVIATE-URL>"
SRC_KEY = "<SOURCE-WEAVIATE-API-KEY>"
# You can set TGT_URL = SRC_URL to migrate within the same instance
TGT_URL = "<TARGET-WEAVIATE-URL>"
TGT_KEY = "<TARGET-WEAVIATE-API-KEY>"

GLOBAL_COLLECTION_NAME = "<COLLECTION-NAME>"  # e.g. "GlobalCollection"

# --- LOGGING ---
logger = logging.getLogger("single_to_mt")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)
    logger.propagate = False


def build_vector_config(config):
    """Re-create the vector config from the template collection. Extend the mapping for
    other vectorizers as needed. Client >= 4.16 uses Configure.Vectors.* with the
    vector_config= parameter (Configure.NamedVectors.* is the legacy vectorizer_config=)."""
    new_vector_config = []
    if config.vector_config:
        for vec_name, vec_data in config.vector_config.items():
            vec_type = vec_data.vectorizer.vectorizer
            logger.info("  vector '%s' -> %s", vec_name, vec_type.value)

            if vec_type == Vectorizers.TEXT2VEC_WEAVIATE:
                new_vector_config.append(
                    Configure.Vectors.text2vec_weaviate(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                        vectorize_collection_name=False,
                    )
                )
            elif vec_type == Vectorizers.TEXT2VEC_OPENAI:
                new_vector_config.append(
                    Configure.Vectors.text2vec_openai(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                    )
                )
            elif vec_type == Vectorizers.TEXT2VEC_COHERE:
                new_vector_config.append(
                    Configure.Vectors.text2vec_cohere(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                    )
                )
            # Add other vectorizers here if needed (Google, Jina, etc.)
            else:
                logger.warning("vectorizer '%s' not mapped; add it to build_vector_config()", vec_type.value)
    return new_vector_config


def migrate():
    # 1. CONNECT
    client_src = weaviate.connect_to_weaviate_cloud(
        cluster_url=SRC_URL,
        auth_credentials=Auth.api_key(SRC_KEY),
        additional_config=wvc.init.AdditionalConfig(timeout=wvc.init.Timeout(init=60, query=120)),
    )

    if SRC_URL == TGT_URL:
        client_tgt = client_src
    else:
        client_tgt = weaviate.connect_to_weaviate_cloud(
            cluster_url=TGT_URL,
            auth_credentials=Auth.api_key(TGT_KEY),
        )

    try:
        # 2. TEMPLATE SCHEMA (from the first source collection)
        src_collections = list(client_src.collections.list_all().keys())
        if not src_collections:
            logger.warning("no collections found on the source cluster")
            return

        template_name = src_collections[0]
        logger.info("using '%s' as schema template", template_name)
        config = client_src.collections.use(template_name).config.get()
        new_vector_config = build_vector_config(config)

        # 3. CREATE GLOBAL MULTI-TENANT COLLECTION
        if client_tgt.collections.exists(GLOBAL_COLLECTION_NAME):
            logger.info("global collection '%s' already exists; reusing it", GLOBAL_COLLECTION_NAME)
            global_coll = client_tgt.collections.use(GLOBAL_COLLECTION_NAME)
        else:
            logger.info("creating global multi-tenant collection '%s'", GLOBAL_COLLECTION_NAME)
            global_coll = client_tgt.collections.create(
                name=GLOBAL_COLLECTION_NAME,
                vector_config=new_vector_config,
                multi_tenancy_config=Configure.multi_tenancy(enabled=True),
            )

        existing_tenants = set(global_coll.tenants.get().keys())

        # 4. MIGRATE EACH COLLECTION INTO ITS OWN TENANT
        for old_name in src_collections:
            if old_name == GLOBAL_COLLECTION_NAME:
                continue

            logger.info("collection '%s' -> tenant '%s'", old_name, old_name)

            if old_name not in existing_tenants:
                global_coll.tenants.create([wvc.tenants.Tenant(name=old_name)])
                existing_tenants.add(old_name)

            src_coll = client_src.collections.use(old_name)
            tgt_tenant = global_coll.with_tenant(old_name)

            count = 0
            with tgt_tenant.batch.fixed_size(batch_size=1000) as batch:
                for obj in tqdm(src_coll.iterator(include_vector=True), desc=old_name, unit="obj"):
                    batch.add_object(
                        properties=obj.properties,
                        vector=obj.vector,
                        uuid=obj.uuid,
                    )
                    count += 1

            failed = tgt_tenant.batch.failed_objects
            if failed:
                logger.error("tenant '%s': %d object(s) failed; first error: %s",
                             old_name, len(failed), failed[0])
            else:
                logger.info("tenant '%s': migrated %d object(s)", old_name, count)

    finally:
        client_src.close()
        if SRC_URL != TGT_URL:
            client_tgt.close()


if __name__ == "__main__":
    migrate()